# 18번. 산업별 충격민감도 (Rolling WLS Beta)

## 개요
**도소매업 산업 전체** 의 거시경제 충격민감도 β 를 추정합니다.
기업별 코드(17번)와 동일한 WLS 방법론을 산업 집계 시계열에 적용합니다.

$$dF_t = \alpha + \beta \cdot dM_t + \varepsilon_t$$

- $dF_t$: 도소매업 산업 평균 매출액증가율
- $dM_t$: GDP 성장률 (거시 충격)
- $\beta$: **산업 충격민감도** — GDP 1%p → 산업 매출 변화 %p

## 기업별 코드와의 차이
| 항목 | 17번 기업별 | 18번 산업별 |
|------|------------|------------|
| 분석 단위 | 기업 개별 | 산업 집계 시계열 |
| 데이터 | 기업 패널 | 단일 시계열 |
| β 개수 | 기업수 × 연도 | 연도당 1개 |
| S2 스코어 | 있음 | 없음 (단일값) |
| Shrinkage | 있음 | 없음 |


In [3]:
import sys, io


import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')


## 1. 파라미터 설정

- `IND_CODE`: 분석 대상 산업 코드 (G = 도매 및 소매업)
- `TEST_YEARS`: 테스트 연도 목록 (2015~2024)
- `DECAY_LAMBDA`: 지수 감쇠율 — 최근 연도에 높은 가중치 부여

In [4]:
BASE       = Path().resolve() / '18번 산업별 충격민감도'
MACRO_CSV  = BASE / '입력데이터/거시지표_통합.csv'
IND_EXCEL  = BASE / '입력데이터/성장성_지표_처리결과.xlsx'

IND_CODE    = 'G'
ITEM_NAME   = '매출액증가율'
TRAIN_START  = 2012
TEST_YEARS   = list(range(2015, 2025))
DECAY_LAMBDA = 0.8

## 2. 헬퍼 함수

### estimate_wls_beta
지수 감쇠 가중치를 적용한 WLS 회귀.
`np.polyfit`에 `sqrt(weights)` 전달 — polyfit 내부가 $\sum(w \cdot \varepsilon)^2$ 최소화이므로 필수.

### granger_test
GDP 성장률(dM)이 매출액증가율(dF)을 Granger 인과하는지 확인.
p < 0.05 → β 추정에 통계적 근거 있음.


In [5]:
def estimate_wls_beta(dM_arr, dF_arr, weights):
    """WLS β 추정 — (β, α, R²) 반환"""
    try:
        coeffs = np.polyfit(dM_arr, dF_arr, 1, w=np.sqrt(weights))
        beta, alpha = float(coeffs[0]), float(coeffs[1])
        if not np.isfinite(beta):
            return None, None, None
        y_pred  = alpha + beta * dM_arr
        w_sum   = weights.sum()
        y_wmean = (weights * dF_arr).sum() / w_sum
        ss_res  = (weights * (dF_arr - y_pred) ** 2).sum()
        ss_tot  = (weights * (dF_arr - y_wmean) ** 2).sum()
        r2      = float(max(0.0, 1 - ss_res / ss_tot)) if ss_tot > 0 else 0.0
        return beta, alpha, r2
    except Exception:
        return None, None, None


def granger_test(ts_dm, ts_df, maxlag=1):
    """dM → dF Granger 인과검정 p값"""
    try:
        res = grangercausalitytests(
            np.column_stack([ts_df, ts_dm]), maxlag=maxlag, verbose=False)
        return float(res[maxlag][0]['ssr_ftest'][1])
    except Exception:
        return np.nan


## 3. 데이터 로드

두 가지 데이터를 연도 기준으로 병합합니다:
1. **GDP 성장률**: 거시지표_통합.csv → 명목 GDP pct_change
2. **산업 매출액증가율**: 성장성_지표_처리결과.xlsx → G 코드 행 추출, wide → long 변환


In [6]:
macro = pd.read_csv(MACRO_CSV, encoding='utf-8-sig')
macro['dM'] = macro['국내총생산(명목, 원화표시)'].pct_change() * 100
macro = macro[['연도', 'dM']].dropna().rename(columns={'연도': 'year'})
macro['year'] = macro['year'].astype(int)
print(f'거시지표: {len(macro)}개 연도  ({macro["year"].min()}~{macro["year"].max()})')

ind_raw = pd.read_excel(IND_EXCEL)
row = ind_raw[
    (ind_raw['코드(업종코드)'] == IND_CODE) &
    (ind_raw['계정항목'] == ITEM_NAME)
]
if row.empty:
    raise ValueError(f'코드={IND_CODE}, 항목={ITEM_NAME} 없음')

year_cols = [c for c in ind_raw.columns if str(c).isdigit()]
dF_series = (
    row[year_cols].T.reset_index()
    .rename(columns={'index': 'year', row.index[0]: 'dF'})
)
dF_series['year'] = dF_series['year'].astype(int)
dF_series['dF']   = pd.to_numeric(dF_series['dF'], errors='coerce')

panel = dF_series.merge(macro, on='year').dropna()
panel = panel[panel['year'] >= TRAIN_START].sort_values('year').reset_index(drop=True)
print(f'산업 데이터: {len(panel)}개 연도  ({panel["year"].min()}~{panel["year"].max()})')
print(panel[['year','dF','dM']].to_string(index=False))


거시지표: 15개 연도  (2012~2026)
산업 데이터: 13개 연도  (2012~2024)
 year    dF       dM
 2012  4.30 3.872666
 2013  2.74 4.403884
 2014  3.44 4.299762
 2015  2.57 6.243036
 2016  5.06 5.299395
 2017 10.33 5.521322
 2018  5.25 3.760693
 2019  2.95 1.675148
 2020  1.38 0.875833
 2021 18.00 7.940202
 2022 12.11 4.584725
 2023 -2.11 3.653782
 2024  2.86 6.151483


## 4. WLS β 추정 (연도별 루프)

각 테스트 연도에 대해:
1. 훈련 데이터: TRAIN_START ~ (TEST_YR - 1)
2. 지수 감쇠 가중치: $w_t = \lambda^{TEST\_YR - t}$
3. WLS 회귀로 β, α, R² 추정
4. Granger 검정으로 통계적 유의성 확인
5. 테스트 연도 실제값과 모형 예측값 비교


In [7]:
all_results = []

for TEST_YR in TEST_YEARS:
    print(f'\n{"=" * 64}')
    print(f'[훈련: {TRAIN_START}~{TEST_YR-1}  |  테스트: {TEST_YR}]')

    train = panel[panel['year'] <= TEST_YR - 1].copy()
    test  = panel[panel['year'] == TEST_YR].copy()
    if test.empty:
        print(f'  {TEST_YR}년 데이터 없음 — 스킵')
        continue

    n_obs     = len(train)
    distances = TEST_YR - train['year'].values
    weights   = DECAY_LAMBDA ** distances

    print(f'훈련 관측수: {n_obs}개  ({train["year"].min()}~{train["year"].max()})')
    print(f'\n연도별 가중치 (λ={DECAY_LAMBDA}):')
    for yr, w in zip(train['year'].values, weights):
        print(f'  {yr}  w={w:.4f}  {"█" * int(w * 20)}')

    beta, alpha, r2 = estimate_wls_beta(train['dM'].values, train['dF'].values, weights)
    if beta is None:
        print('  β 추정 실패')
        continue

    p_granger = granger_test(train['dM'].values, train['dF'].values)
    actual_dF = float(test['dF'].values[0])
    actual_dM = float(test['dM'].values[0])
    pred_dF   = alpha + beta * actual_dM

    print(f'\n  β={beta:.4f}  α={alpha:.4f}  R²={r2:.4f}  Granger_p={p_granger:.4f}')
    print(f'  실제 dM={actual_dM:.4f}%  실제 dF={actual_dF:.4f}%  예측={pred_dF:.4f}%  잔차={actual_dF-pred_dF:.4f}%p')

    all_results.append({
        '산업명': '도매 및 소매업', '코드': IND_CODE,
        '테스트_연도': TEST_YR, '훈련_시작': TRAIN_START, '훈련_끝': TEST_YR-1,
        'n_obs': n_obs, 'beta_i': round(beta,6), 'alpha': round(alpha,6),
        'r2': round(r2,6), 'decay_lambda': DECAY_LAMBDA,
        'granger_p': round(p_granger,4), 'actual_dM': round(actual_dM,4),
        'actual_dF': round(actual_dF,4), 'pred_dF': round(pred_dF,4),
    })



[훈련: 2012~2014  |  테스트: 2015]
훈련 관측수: 3개  (2012~2014)

연도별 가중치 (λ=0.8):
  2012  w=0.5120  ██████████
  2013  w=0.6400  ████████████
  2014  w=0.8000  ████████████████

  β=-2.6591  α=14.6623  R²=0.9039  Granger_p=nan
  실제 dM=6.2430%  실제 dF=2.5700%  예측=-1.9383%  잔차=4.5083%p

[훈련: 2012~2015  |  테스트: 2016]
훈련 관측수: 4개  (2012~2015)

연도별 가중치 (λ=0.8):
  2012  w=0.4096  ████████
  2013  w=0.5120  ██████████
  2014  w=0.6400  ████████████
  2015  w=0.8000  ████████████████

  β=-0.4993  α=5.5923  R²=0.5852  Granger_p=nan
  실제 dM=5.2994%  실제 dF=5.0600%  예측=2.9466%  잔차=2.1134%p

[훈련: 2012~2016  |  테스트: 2017]
훈련 관측수: 5개  (2012~2016)

연도별 가중치 (λ=0.8):
  2012  w=0.3277  ██████
  2013  w=0.4096  ████████
  2014  w=0.5120  ██████████
  2015  w=0.6400  ████████████
  2016  w=0.8000  ████████████████

  β=-0.2500  α=4.9690  R²=0.0414  Granger_p=0.2821
  실제 dM=5.5213%  실제 dF=10.3300%  예측=3.5886%  잔차=6.7414%p

[훈련: 2012~2017  |  테스트: 2018]
훈련 관측수: 6개  (2012~2017)

연도별 가중치 (λ=0.8):
  2012  w=0.2621  █████

## 5. 저장

In [8]:
result  = pd.DataFrame(all_results)
out_csv = BASE / '산업충격민감도_OLS.csv'
result.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'저장 완료: {out_csv}')
print(result[['테스트_연도','beta_i','r2','granger_p','actual_dF','pred_dF']].to_string(index=False))


저장 완료: C:\Users\ieoql\Documents\취업준비\code\Corporate Bankruptcy\corporate-bankruptcy\데이터 전처리\18번 산업별 충격민감도\산업충격민감도_OLS.csv
 테스트_연도    beta_i       r2  granger_p  actual_dF  pred_dF
   2015 -2.659061 0.903851        NaN       2.57  -1.9383
   2016 -0.499253 0.585203        NaN       5.06   2.9466
   2017 -0.250013 0.041402     0.2821      10.33   3.5886
   2018  0.941866 0.052418     0.1263       5.25   4.1902
   2019  0.585123 0.038136     0.3804       2.95   3.6105
   2020  0.743401 0.203141     0.2396       1.38   2.4813
   2021  0.914785 0.426303     0.1396      18.00   8.2706
   2022  2.112482 0.739443     0.5098      12.11   7.6571
   2023  2.146830 0.673726     0.4760      -2.11   6.6390
   2024  2.416632 0.526547     0.8621       2.86  10.6370
